In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import linregress

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR = BASE_DIR

fund_master = pd.read_csv(DATA_DIR / '01_fund_master.csv')
nav_history = pd.read_csv(DATA_DIR / '02_nav_history.csv')
performance = pd.read_csv(DATA_DIR / '07_scheme_performance.csv')
benchmarks = pd.read_csv(DATA_DIR / '10_benchmark_indices.csv')

print('Loaded datasets successfully.')

In [ ]:
# Standardize fund identifiers and dates
fund_master['amfi_code'] = pd.to_numeric(fund_master['amfi_code'], errors='coerce')
nav_history['amfi_code'] = pd.to_numeric(nav_history['amfi_code'], errors='coerce')
nav_history['date'] = pd.to_datetime(nav_history['date'], errors='coerce')
nav_history['nav'] = pd.to_numeric(nav_history['nav'], errors='coerce')
nav_history = nav_history.dropna(subset=['amfi_code', 'date', 'nav']).copy()

# Build daily panel with forward-fill for missing NAVs
panel_frames = []
for amfi_code, group in nav_history.groupby('amfi_code', sort=False):
    group = group.sort_values('date')
    full_index = pd.date_range(group['date'].min(), group['date'].max(), freq='D')
    filled = group.set_index('date').reindex(full_index)
    filled['nav'] = filled['nav'].ffill().bfill()
    filled['amfi_code'] = int(amfi_code)
    filled = filled.reset_index().rename(columns={'index': 'date'})
    panel_frames.append(filled)

nav_panel = pd.concat(panel_frames, ignore_index=True)
nav_panel = nav_panel[['amfi_code', 'date', 'nav']].copy()
nav_panel = nav_panel.merge(
    fund_master[['amfi_code', 'scheme_name', 'fund_house', 'expense_ratio_pct']],
    on='amfi_code',
    how='left',
)

# Standardize benchmarks
benchmarks['date'] = pd.to_datetime(benchmarks['date'], errors='coerce')
benchmarks['index_name'] = benchmarks['index_name'].astype(str).str.upper().str.replace(' ', '', regex=False)
benchmarks['close_value'] = pd.to_numeric(benchmarks['close_value'], errors='coerce')
benchmarks = benchmarks.dropna(subset=['date', 'close_value']).copy()
benchmark_wide = benchmarks.pivot_table(index='date', columns='index_name', values='close_value').sort_index()
benchmark_wide = benchmark_wide[['NIFTY50', 'NIFTY100']].copy()

nav_panel.head(), benchmark_wide.head()

In [ ]:
# Daily returns for all funds
nav_panel = nav_panel.sort_values(['amfi_code', 'date']).copy()
nav_panel['prev_nav'] = nav_panel.groupby('amfi_code')['nav'].shift(1)
nav_panel['daily_return'] = nav_panel['nav'] / nav_panel['prev_nav'] - 1
returns_df = nav_panel.pivot_table(index='date', columns='amfi_code', values='daily_return').sort_index()
returns_df = returns_df.dropna(how='all')

# Basic validation summary
summary = returns_df.apply(lambda s: s.dropna().agg(['count', 'mean', 'median', 'std', 'skew', 'kurt'])).T
print(summary.head())

# Sample histograms
sample_funds = returns_df.columns[:3]
for fund in sample_funds:
    plt.figure(figsize=(6, 3))
    returns_df[fund].dropna().hist(bins=50, alpha=0.7)
    plt.title(f'Histogram of daily returns for fund {fund}')
    plt.xlabel('Daily return')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()


In [ ]:
# CAGR values for 1y, 3y, 5y
cagr_rows = []
for amfi_code, group in nav_panel.groupby('amfi_code', sort=False):
    group = group.sort_values('date')
    end_nav = group['nav'].iloc[-1]
    for years in [1, 3, 5]:
        start_date = group['date'].max() - pd.DateOffset(years=years)
        start_nav = group.loc[group['date'] >= start_date, 'nav'].iloc[0]
        cagr = (end_nav / start_nav) ** (1 / years) - 1 if start_nav > 0 else np.nan
        cagr_rows.append({'amfi_code': int(amfi_code), 'years': years, 'cagr': cagr})

cagr_df = pd.DataFrame(cagr_rows).pivot(index='amfi_code', columns='years', values='cagr').rename(columns={1: 'cagr_1y', 3: 'cagr_3y', 5: 'cagr_5y'})
print(cagr_df.head())

In [ ]:
rf = 0.065
annualization = 252

# Sharpe and Sortino ratios
mean_daily = returns_df.mean()
std_daily = returns_df.std()
sharpe = ((mean_daily - (rf / annualization)) / std_daily) * np.sqrt(annualization)

downside = returns_df.apply(lambda s: s[s < 0].std() if (s < 0).any() else np.nan)
sortino = ((mean_daily - (rf / annualization)) / downside) * np.sqrt(annualization)

sharpe_ranked = pd.DataFrame({'sharpe_ratio': sharpe, 'sortino_ratio': sortino}).reset_index().rename(columns={'index': 'amfi_code'})
sharpe_ranked['sharpe_rank'] = sharpe_ranked['sharpe_ratio'].rank(ascending=False, method='average')
sharpe_ranked['sortino_rank'] = sharpe_ranked['sortino_ratio'].rank(ascending=False, method='average')
sharpe_ranked.head()

In [ ]:
# Alpha and beta vs Nifty 100 daily excess returns
benchmark_returns = benchmark_wide['NIFTY100'].pct_change().dropna()
alpha_beta_rows = []
for amfi_code in returns_df.columns:
    fund_returns = returns_df[amfi_code].dropna()
    bench = benchmark_returns.reindex(fund_returns.index).dropna()
    common = pd.concat([fund_returns.rename('fund'), bench.rename('benchmark')], axis=1).dropna()
    if len(common) < 3:
        continue
    slope, intercept, r_value, p_value, stderr = linregress(common['benchmark'], common['fund'])
    alpha_beta_rows.append({
        'amfi_code': int(amfi_code),
        'alpha': intercept * annualization,
        'beta': slope,
        'r_value': r_value,
        'slope_pval': p_value,
        'intercept_pval': p_value,
        'stderr': stderr,
    })

alpha_beta = pd.DataFrame(alpha_beta_rows)
alpha_beta.head()

In [ ]:
# Max drawdown and worst dates
max_dd_rows = []
for amfi_code, group in nav_panel.groupby('amfi_code', sort=False):
    group = group.sort_values('date').copy()
    group['running_max'] = group['nav'].cummax()
    group['drawdown'] = group['nav'] / group['running_max'] - 1
    worst_idx = group['drawdown'].idxmin()
    worst = group.loc[worst_idx]
    max_dd_rows.append({
        'amfi_code': int(amfi_code),
        'max_drawdown': worst['drawdown'],
        'max_drawdown_start_date': worst['date'] if worst['drawdown'] == group['drawdown'].min() else None,
        'max_drawdown_end_date': worst['date'],
    })

max_dd = pd.DataFrame(max_dd_rows)
max_dd.head()

In [ ]:
# Prepare scorecard
fund_meta = fund_master[['amfi_code', 'scheme_name', 'fund_house', 'expense_ratio_pct']].rename(columns={'scheme_name': 'fund_name', 'expense_ratio_pct': 'expense_ratio'})
metrics = cagr_df.reset_index().merge(sharpe_ranked, on='amfi_code', how='left').merge(alpha_beta, on='amfi_code', how='left').merge(max_dd, on='amfi_code', how='left').merge(fund_meta, on='amfi_code', how='left')
metrics['expense_ratio_rank'] = metrics['expense_ratio'].rank(ascending=True, method='average')
metrics['max_drawdown_rank'] = metrics['max_drawdown'].abs().rank(ascending=True, method='average')
metrics['return_3y_rank'] = metrics['cagr_3y'].rank(ascending=False, method='average')
metrics['alpha_rank'] = metrics['alpha'].rank(ascending=False, method='average')
metrics['sharpe_rank'] = metrics['sharpe_ratio'].rank(ascending=False, method='average')
metrics['composite_score'] = (
    0.30 * metrics['return_3y_rank'] +
    0.25 * metrics['sharpe_rank'] +
    0.20 * metrics['alpha_rank'] +
    0.15 * metrics['expense_ratio_rank'] +
    0.10 * metrics['max_drawdown_rank']
)
metrics['composite_score'] = 100 * (len(metrics) - metrics['composite_score']) / (len(metrics) - 1)
metrics['composite_rank'] = metrics['composite_score'].rank(ascending=False, method='average')
scorecard = metrics.sort_values('composite_score', ascending=False).reset_index(drop=True)

scorecard[['fund_name', 'composite_score', 'composite_rank']].head(10)

scorecard.to_csv(OUTPUT_DIR / 'fund_scorecard.csv', index=False)
alpha_beta.to_csv(OUTPUT_DIR / 'alpha_beta.csv', index=False)

In [ ]:
# Benchmark comparison for top 5 funds over 3 years
latest_date = nav_panel['date'].max()
window_start = latest_date - pd.DateOffset(years=3)

selected_funds = scorecard.head(5)['amfi_code'].astype(int).tolist()
plot_frames = []
for amfi_code in selected_funds:
    fund_series = nav_panel.loc[nav_panel['amfi_code'] == amfi_code, ['date', 'nav']].sort_values('date')
    fund_series = fund_series.set_index('date')['nav'].loc[window_start:latest_date]
    fund_series = fund_series.ffill().bfill()
    if not fund_series.empty:
        normalized = fund_series / fund_series.iloc[0] * 100
        normalized.name = scorecard.loc[scorecard['amfi_code'] == amfi_code, 'fund_name'].iloc[0]
        plot_frames.append(normalized)

for benchmark_name in ['NIFTY50', 'NIFTY100']:
    bench_series = benchmark_wide[benchmark_name].sort_index().loc[window_start:latest_date].ffill().bfill()
    if not bench_series.empty:
        normalized = bench_series / bench_series.iloc[0] * 100
        normalized.name = benchmark_name
        plot_frames.append(normalized)

comparison_df = pd.concat(plot_frames, axis=1).dropna()

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(14, 8))
for col in comparison_df.columns:
    ax.plot(comparison_df.index, comparison_df[col], label=col)
ax.set_title('Top 5 Funds vs Nifty 50/100 over 3 Years')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized NAV (100 = start)')
ax.legend(loc='best')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'benchmark_comparison.png', dpi=300, bbox_inches='tight')
plt.close(fig)

print('Saved benchmark comparison chart.')

## Validation Checklist

- Daily return distributions generated and reviewed.
- CAGR comparison table created.
- Sharpe and Sortino ranks computed.
- Alpha/Beta outputs saved to CSV.
- Fund scorecard saved to CSV.
- Benchmark comparison chart saved as PNG.

# Mutual Fund Performance Analytics

This notebook implements the end-to-end performance analytics workflow for mutual funds, including daily return calculations, CAGR, Sharpe and Sortino ratios, alpha/beta regression, maximum drawdown, scorecard generation, benchmark comparison, and CSV/PNG outputs.